# Performance Figures

Hourly RTT, loss, second-granularity event traces, and RTT difference CDFs for the obstructed and unobstructed measurement periods.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'process'))

# Processing scripts (IRTT, TLE, pipeline): see process/
from data_loading import *
from satellite_matching import *
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
from tqdm import tqdm


## Hourly RTT and Loss

The SE-obstructed period (Jan 17–27) and unobstructed baseline (Jan 29 – Feb 7) are each loaded below.

Run each pair of load + plot cells for the desired period by setting `start_date` / `end_date`.

### SE-Obstructed Period (Jan 17–27)

In [ ]:
start_date = "2025-01-17"
end_date   = "2025-01-27"

rtt_data_df = read_rtt_data(start_date, end_date)

hourly_rtt = rtt_data_df.rename(columns={'date': 'timestamp'}).copy()
hourly_rtt['timestamp'] = hourly_rtt['timestamp'].dt.floor('h')
hourly_rtt = hourly_rtt.groupby('timestamp').agg(
    rtt_pi1_mean   = ('rtt_pi1', 'mean'),
    rtt_pi1_median = ('rtt_pi1', 'median'),
    rtt_pi1_p05    = ('rtt_pi1', lambda x: x.quantile(0.05)),
    rtt_pi1_p95    = ('rtt_pi1', lambda x: x.quantile(0.95)),
    rtt_pi2_mean   = ('rtt_pi2', 'mean'),
    rtt_pi2_median = ('rtt_pi2', 'median'),
    rtt_pi2_p05    = ('rtt_pi2', lambda x: x.quantile(0.05)),
    rtt_pi2_p95    = ('rtt_pi2', lambda x: x.quantile(0.95)),
).reset_index()
print(hourly_rtt.head())

In [ ]:
# hourly RTT with p05–p95 shaded bands
pi2_color = 'tab:orange'
fig, ax = plt.subplots(figsize=(6, 3))

ax.fill_between(hourly_rtt['timestamp'], hourly_rtt['rtt_pi1_p05'], hourly_rtt['rtt_pi1_p95'],
                alpha=0.3, color='blue', label='Control RTT\nP05-P95 Range')
ax.fill_between(hourly_rtt['timestamp'], hourly_rtt['rtt_pi2_p05'], hourly_rtt['rtt_pi2_p95'],
                alpha=0.3, color=pi2_color, label='Test RTT\nP05-P95 Range')

ax.plot(hourly_rtt['timestamp'], hourly_rtt['rtt_pi1_median'], color='darkblue', label='Control RTT\nMedian')
ax.plot(hourly_rtt['timestamp'], hourly_rtt['rtt_pi2_median'], color=pi2_color,  label='Test RTT\nMedian')

ax.plot(hourly_rtt['timestamp'], hourly_rtt['rtt_pi1_p05'], color='blue',    linewidth=1, alpha=0.7, linestyle='--')
ax.plot(hourly_rtt['timestamp'], hourly_rtt['rtt_pi1_p95'], color='blue',    linewidth=1, alpha=0.7, linestyle='--')
ax.plot(hourly_rtt['timestamp'], hourly_rtt['rtt_pi2_p05'], color=pi2_color, linewidth=1, alpha=0.7, linestyle='--')
ax.plot(hourly_rtt['timestamp'], hourly_rtt['rtt_pi2_p95'], color=pi2_color, linewidth=1, alpha=0.7, linestyle='--')

ax.set_ylim(30, 80)
ax.set_xlabel('Time')
ax.set_ylabel('RTT (ms)')
ax.grid(True, alpha=0.3)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.xaxis.set_major_locator(mdates.HourLocator(interval=24))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.xticks(rotation=45)
plt.tight_layout()
# plt.savefig('figures/rtt_obstructed.pdf', bbox_inches='tight')
plt.show()

In [ ]:
start_date = "2025-01-17"
end_date   = "2025-01-26"

loss_data = get_loss_data(start_date, end_date)
loss_df = loss_data.rename(columns={'date': 'timestamp'}).copy()
loss_df['timestamp'] = loss_df['timestamp'].dt.floor('h')
loss_df = loss_df.groupby('timestamp').agg(
    pi1_loss_sum=('pi1_loss', 'sum'),
    pi2_loss_sum=('pi2_loss', 'sum'),
).reset_index()

In [ ]:
total_loss = 5 * 100 * 60 * 60

fig, ax = plt.subplots(figsize=(5, 3))
plt.rc('font', size=11)
ax.plot(loss_df['timestamp'], loss_df['pi1_loss_sum'] / (total_loss / 100), label='Control Loss')
ax.plot(loss_df['timestamp'], -loss_df['pi2_loss_sum'] / (total_loss / 100), label='Test Loss')
ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f"{abs(x):.0f}"))
plt.xticks(rotation=45)
ax.set_xlabel('Time')
ax.set_ylabel('Loss (%)')
ax.legend(loc='upper right')
ax.grid()
plt.tight_layout()
# plt.savefig('figures/loss_obstructed.pdf', bbox_inches='tight')
plt.show()

### Unobstructed Baseline (Jan 29 – Feb 7)

In [ ]:
start_date = "2025-01-29"
end_date   = "2025-02-07"

rtt_data_df = read_rtt_data(start_date, end_date)

hourly_rtt = rtt_data_df.rename(columns={'date': 'timestamp'}).copy()
hourly_rtt['timestamp'] = hourly_rtt['timestamp'].dt.floor('h')
hourly_rtt = hourly_rtt.groupby('timestamp').agg(
    rtt_pi1_mean   = ('rtt_pi1', 'mean'),
    rtt_pi1_median = ('rtt_pi1', 'median'),
    rtt_pi1_p05    = ('rtt_pi1', lambda x: x.quantile(0.05)),
    rtt_pi1_p95    = ('rtt_pi1', lambda x: x.quantile(0.95)),
    rtt_pi2_mean   = ('rtt_pi2', 'mean'),
    rtt_pi2_median = ('rtt_pi2', 'median'),
    rtt_pi2_p05    = ('rtt_pi2', lambda x: x.quantile(0.05)),
    rtt_pi2_p95    = ('rtt_pi2', lambda x: x.quantile(0.95)),
).reset_index()
# Drop last 12 hours (incomplete data at period boundary)
hourly_rtt = hourly_rtt[hourly_rtt['timestamp'] < hourly_rtt['timestamp'].max() - pd.Timedelta(hours=12)]
print(hourly_rtt.head())

In [ ]:
# hourly RTT with p05–p95 shaded bands
pi2_color = 'tab:orange'
fig, ax = plt.subplots(figsize=(6, 3))

ax.fill_between(hourly_rtt['timestamp'], hourly_rtt['rtt_pi1_p05'], hourly_rtt['rtt_pi1_p95'],
                alpha=0.3, color='blue', label='Control RTT\nP05-P95 Range')
ax.fill_between(hourly_rtt['timestamp'], hourly_rtt['rtt_pi2_p05'], hourly_rtt['rtt_pi2_p95'],
                alpha=0.3, color=pi2_color, label='Test RTT\nP05-P95 Range')

ax.plot(hourly_rtt['timestamp'], hourly_rtt['rtt_pi1_median'], color='darkblue', label='Control RTT\nMedian')
ax.plot(hourly_rtt['timestamp'], hourly_rtt['rtt_pi2_median'], color=pi2_color,  label='Test RTT\nMedian')

ax.plot(hourly_rtt['timestamp'], hourly_rtt['rtt_pi1_p05'], color='blue',    linewidth=1, alpha=0.7, linestyle='--')
ax.plot(hourly_rtt['timestamp'], hourly_rtt['rtt_pi1_p95'], color='blue',    linewidth=1, alpha=0.7, linestyle='--')
ax.plot(hourly_rtt['timestamp'], hourly_rtt['rtt_pi2_p05'], color=pi2_color, linewidth=1, alpha=0.7, linestyle='--')
ax.plot(hourly_rtt['timestamp'], hourly_rtt['rtt_pi2_p95'], color=pi2_color, linewidth=1, alpha=0.7, linestyle='--')

ax.set_ylim(30, 80)
ax.set_xlabel('Time')
ax.set_ylabel('RTT (ms)')
ax.grid(True, alpha=0.3)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.xaxis.set_major_locator(mdates.HourLocator(interval=24))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.xticks(rotation=45)
plt.tight_layout()
# plt.savefig('figures/rtt_unobstructed.pdf', bbox_inches='tight')
plt.show()

In [ ]:
start_date = "2025-01-29"
end_date   = "2025-02-07"

loss_data = get_loss_data(start_date, end_date)
loss_df = loss_data.rename(columns={'date': 'timestamp'}).copy()
loss_df['timestamp'] = loss_df['timestamp'].dt.floor('h')
loss_df = loss_df.groupby('timestamp').agg(
    pi1_loss_sum=('pi1_loss', 'sum'),
    pi2_loss_sum=('pi2_loss', 'sum'),
).reset_index()

In [ ]:
total_loss = 5 * 100 * 60 * 60

fig, ax = plt.subplots(figsize=(5, 3))
plt.rc('font', size=11)
ax.plot(loss_df['timestamp'], loss_df['pi1_loss_sum'] / (total_loss / 100), label='Control Loss')
ax.plot(loss_df['timestamp'], -loss_df['pi2_loss_sum'] / (total_loss / 100), label='Test Loss')
ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f"{abs(x):.0f}"))
plt.xticks(rotation=45)
ax.set_xlabel('Time')
ax.set_ylabel('Loss (%)')
ax.legend(loc='upper right')
ax.grid()
plt.tight_layout()
# plt.savefig('figures/loss_unobstructed.pdf', bbox_inches='tight')
plt.show()

## Second-Granularity RTT + Loss Spike

Shows a specific responsive routing event on Jan 1, 2025 where both dishes briefly connect to different satellites. The event is identified by finding contiguous windows where pi1_sat ≠ pi2_sat, then selecting the 22nd such chunk (index 21).

In [ ]:
est = timezone('US/Eastern')
event_start = datetime(2025, 1, 1, 0, 0, 0, 0, est)
event_end   = datetime(2025, 1, 1, 23, 59, 59, 0, est)

rst_obsmap_dict = get_rst_obsmap_dict(event_start, event_end)
rtt_data_df     = read_rtt_data("2025-01-01", "2025-01-01")
loss_data       = get_loss_data("2025-01-01", "2025-01-01")
sat_match_data  = read_sat_match_data("2025-01-01", "2025-01-03")
all_sat_data    = sat_match_data.sort_values("date").reset_index(drop=True)

In [ ]:
# Find contiguous diff-sat event chunks
diff_indices = all_sat_data.index[all_sat_data["pi1_sat"] != all_sat_data["pi2_sat"]]
extended = set(diff_indices)
for idx in diff_indices:
    if idx > 0: extended.add(idx - 1)
    if idx < len(all_sat_data) - 1: extended.add(idx + 1)

diff_sat_data = all_sat_data.loc[sorted(extended)].reset_index(drop=True)

chunks, current_chunk = [], []
for _, row in diff_sat_data.iterrows():
    if not current_chunk:
        current_chunk.append(row['date'])
    else:
        current_chunk.append(row['date'])
        if row['pi1_sat'] == row['pi2_sat']:
            chunks.append(current_chunk)
            current_chunk = []

# Keep only fully contiguous chunks (no gaps)
filtered_chunks = []
for chunk in chunks:
    contiguous = all(
        pd.to_datetime(chunk[i]) + pd.Timedelta(seconds=15) == pd.to_datetime(chunk[i+1])
        for i in range(len(chunk)-1)
    )
    if contiguous:
        filtered_chunks.append(chunk)

print(f"Found {len(filtered_chunks)} contiguous diff-sat event chunks")

In [ ]:
# plot event chunk 21 (index 21) at second granularity
time_chunks = filtered_chunks[21]
event_date  = time_chunks[1]  # middle window of the chunk

start_time = event_date - pd.Timedelta(seconds=16)
end_time   = event_date

rtt_chunk  = rtt_data_df[(rtt_data_df['date'] > start_time) & (rtt_data_df['date'] < end_time)]
loss_chunk = loss_data[(loss_data['date'] > start_time) & (loss_data['date'] < end_time)]

# Align loss to RTT timestamps
aligned_loss = pd.merge(rtt_chunk[['date']], loss_chunk[['date', 'pi1_loss', 'pi2_loss']],
                        on='date', how='left').fillna(0)

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(rtt_chunk['date'], rtt_chunk['rtt_pi1'], color='tab:blue',   label='Control RTT')
ax.plot(rtt_chunk['date'], rtt_chunk['rtt_pi2'], color='tab:orange', label='Test RTT')
ax.axvline(pd.Timestamp("2025-01-01 07:01:12-05:00"), color='black',               label='Handover 1')
ax.axvline(pd.Timestamp("2025-01-01 07:01:16-05:00") + pd.Timedelta(milliseconds=200),
           color='black', linestyle='--', label='Handover 2')

ax2 = ax.twinx()
ax2.scatter(aligned_loss['date'], aligned_loss['pi1_loss'], color='tab:cyan', label='Control Loss', s=3)
ax2.scatter(aligned_loss['date'], aligned_loss['pi2_loss'], color='tab:red',  label='Test Loss',    s=3)
ax2.set_ylim(0, 6)
ax2.set_ylabel('Loss')

ax.xaxis.set_major_locator(mdates.SecondLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%S'))
ax.set_xlabel('Timestamp (seconds)')
ax.set_ylabel('RTT (ms)')
ax.grid()

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8)
plt.tight_layout()
# plt.savefig('figures/rtt_loss_spike.pdf', bbox_inches='tight')
plt.show()

show_obs_image(time_chunks[0], rst_obsmap_dict)

## CDF of RTT Difference — Same vs Different Satellite

Compares the distribution of (Control − Test) RTT for 15-second windows where both dishes are on the **same** satellite vs **different** satellites. Uses the SE-obstructed period (Jan 17–27, 2025).

RTT measurements are bucketed into 15-second intervals matching the sat-match timestamps, then the median and P05 differences are plotted.

In [ ]:
start_date = "2025-01-17"
end_date   = "2025-01-27"

sat_match_data = read_sat_match_data(start_date, end_date)
rtt_data_df    = read_rtt_data(start_date, end_date)

same_sat = sat_match_data[sat_match_data['pi1_sat'] == sat_match_data['pi2_sat']]['date'].tolist()
diff_sat = sat_match_data[sat_match_data['pi1_sat'] != sat_match_data['pi2_sat']]['date'].tolist()
print(f"Same satellite periods: {len(same_sat)}")
print(f"Different satellite periods: {len(diff_sat)}")

In [ ]:
# Bucket each RTT measurement into its 15-second sat-match window
rtt = rtt_data_df.rename(columns={'date': 'timestamp'}).copy()
rtt['timestamp'] = pd.to_datetime(rtt['timestamp'])

def assign_bucket(ts):
    s = ts.second
    base = ts.floor('min')
    if s < 11:   return (ts - pd.Timedelta(seconds=s+1)).floor('min') + pd.Timedelta(seconds=56)
    elif s < 26: return base + pd.Timedelta(seconds=11)
    elif s < 41: return base + pd.Timedelta(seconds=26)
    elif s < 56: return base + pd.Timedelta(seconds=41)
    else:        return base + pd.Timedelta(seconds=56)

rtt['timestamp'] = rtt['timestamp'].apply(assign_bucket)

irtt_pdf = rtt.groupby('timestamp').agg(
    rtt_pi1_median=('rtt_pi1', 'median'),
    rtt_pi1_p05   =('rtt_pi1', lambda x: x.quantile(0.05)),
    rtt_pi2_median=('rtt_pi2', 'median'),
    rtt_pi2_p05   =('rtt_pi2', lambda x: x.quantile(0.05)),
).reset_index()

irtt_pdf['diff_median'] = irtt_pdf['rtt_pi1_median'] - irtt_pdf['rtt_pi2_median']
irtt_pdf['diff_p05']    = irtt_pdf['rtt_pi1_p05']    - irtt_pdf['rtt_pi2_p05']

irtt_same = irtt_pdf[irtt_pdf['timestamp'].isin(same_sat)]
irtt_diff = irtt_pdf[irtt_pdf['timestamp'].isin(diff_sat)]
print(f"Same-sat RTT rows: {len(irtt_same)}, Diff-sat RTT rows: {len(irtt_diff)}")

In [ ]:
# CDF of RTT difference (control − test)
fig, ax = plt.subplots(figsize=(7, 3))

for data, color, sat_label in [(irtt_same, 'tab:blue', 'Same Satellites'),
                                (irtt_diff, 'tab:orange', 'Different Satellites')]:
    for col, style, stat_label in [('diff_median', '-', 'Median'), ('diff_p05', '--', 'P05')]:
        vals = data[col].dropna()
        if len(vals) == 0:
            continue
        sorted_vals = np.sort(vals)
        ax.plot(sorted_vals, np.linspace(0, 1, len(sorted_vals)),
                color=color, linestyle=style, linewidth=2,
                label=f'{sat_label}\n({stat_label})')

ax.set_xlabel('RTT Difference (Control - Test) [ms]')
ax.set_ylabel('Cumulative Probability')
ax.set_xlim(-20, 20)
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left', bbox_to_anchor=(1.05, 1.0), frameon=True)
plt.tight_layout()
# plt.savefig('figures/rtt_diff_cdf.pdf', bbox_inches='tight')
plt.show()